In [1]:
import openeo
import rasterio
from localtileserver import TileClient, get_leaflet_tile_layer
import leafmap

In [2]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

#  Importing Shapefile from Syke
https://www.syke.fi/en/environmental-data/downloadable-spatial-datasets#peat-production-areas-after-use-category

In [10]:
import geopandas as gpd

In [11]:
peat = gpd.read_file("../data/raw/vectors/turvetuotantoalueet/turvetuotantoalueet_jalkikaytto.shp")

print(peat.crs)
print(len(peat))
print(peat.columns.tolist())
peat.head()

PROJCS["EUREF_FIN_TM35FIN",GEOGCS["EUREF-FIN",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",27],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
164821
['OBJECTID', 'luokka', 'luokka_nro', 'Shape_Leng', 'Shape_Le_1', 'Shape_Area', 'geometry']


,OBJECTID,luokka,luokka_nro,Shape_Leng,Shape_Le_1,Shape_Area,geometry
0,1,avoin,5,19.387451,19.387451,22.538417,"POLYGON ((297344 6657056, 297344 6657051.677, ..."
1,2,avoin,5,57.283742,57.283742,55.436297,"POLYGON ((297336 6657080, 297336 6657063.652, ..."
2,3,avoin,5,29.051756,29.051756,20.186929,"POLYGON ((297376 6657088, 297363.251 6657088, ..."
3,4,avoin,5,24.240326,24.240326,24.311072,"POLYGON ((297392 6657096, 297383.813 6657096, ..."
4,5,avoin,5,15.272810,15.272810,9.650851,"POLYGON ((297400 6657104, 297394.842 6657104, ..."


In [ ]:
#reproject to WGS84 for visualization
peat_wgs = peat.to_crs("EPSG:4326")
# visualising all peat areas in Finland dataset last updated 28.03.2023
# former peat production areas and the classification of what they are today
m = leafmap.Map(center=[63.0, 25.0], zoom=6)
m.add_gdf(peat_wgs, layer_name="Peat production areas")
m

Map(center=[63.0, 25.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

In [14]:
#reproject to WGS84 for visualization
peat_wgs = peat.to_crs("EPSG:4326")
# South Ostrobothnia and the area reaching south towards Satakunta and Pirkanmaa
study_area = {"west": 21.8, "south": 61.3, "east": 24.3, "north": 63.3}

peat_sub = peat_wgs.cx[study_area["west"]:study_area["east"], study_area["south"]:study_area["north"]]
print(f"{len(peat_sub)} peat production areas in the study area")

m = leafmap.Map(center=[62.3, 23.0], zoom=8)
m.add_gdf(peat_sub, layer_name="Peat production areas")
m

36359 peat production areas in the study area


Map(center=[62.3, 23.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

# Visualising Sentinel 2

In [17]:
# smaller test area, adjust to wherever the peat polygons cluster
test_area = {"west": 22.4, "south": 62.3, "east": 22.8, "north": 62.6}

finland_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=test_area,
    temporal_extent=["2024-06-01", "2024-08-31"],
    bands=["B04", "B03", "B02"],
    max_cloud_cover=30,
)

finland_composite = finland_cube.reduce_dimension(dimension="t", reducer="median")

finland_composite.download("../data/raw/raster/finland_test_rgb_2024.tif")

In [18]:
from localtileserver import TileClient, get_leaflet_tile_layer
import leafmap

In [19]:
client = TileClient("../data/raw/raster/finland_test_rgb_2024.tif", host="127.0.0.1")
layer = get_leaflet_tile_layer(client, band=[1, 2, 3], vmin=0, vmax=3000)

m = leafmap.Map(center=[62.45, 22.6], zoom=11)
m.add_layer(layer)
m
m

Map(center=[62.45, 22.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [20]:
peat_test = peat_wgs.cx[22.4:22.8, 62.3:62.6]

m.add_gdf(peat_test, layer_name="Peat sites", style={"fillOpacity": 0, "color": "red", "weight": 1})
m

Map(bottom=72768.0, center=[62.41835992894367, 22.64419555664063], controls=(ZoomControl(options=['position', …